In [11]:
# ============================================================
# 1️⃣ IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import seaborn as sns
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, precision_score, recall_score, f1_score, accuracy_score, confusion_matrix

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, SimpleRNN
from tensorflow.keras.callbacks import EarlyStopping

from tabulate import tabulate

In [12]:
# ============================================================
# 2️⃣ LOAD & PREPROCESS MULTI-YEAR NSE DATASET
# ============================================================

df = pd.read_csv("NIFTY_5_Years.csv", encoding="utf-8-sig")

# Clean columns
df.columns = df.columns.str.strip()

# Convert date
df['Date'] = pd.to_datetime(df['Date'])

# Sort ascending
df = df.sort_values("Date")

# Set index
df.set_index("Date", inplace=True)

# Use Close price
data = df[['Close']]

# Train-Test Split (80-20)
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

print("Training size:", len(train))
print("Testing size :", len(test))

Training size: 992
Testing size : 248


In [13]:
# ============================================================
# 4️⃣ HELPER FUNCTIONS
# ============================================================

def create_sequences(data, time_steps=60):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:i+time_steps])
        y.append(data[i+time_steps])
    return np.array(X), np.array(y)


def evaluate_model(actual, predicted, name):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)

    print(f"\n{name}")
    print("RMSE:", rmse)
    print("MAE :", mae)
    print("R2  :", r2)

    return rmse, mae, r2


time_steps = 60

In [14]:
import numpy as np

def predict_future_prices(model, last_data, scaler, time_steps, future_days):
    
    # Ensure correct shape
    input_seq = last_data[-time_steps:]
    input_seq = input_seq.reshape(1, time_steps, 1)

    future_predictions = []

    for _ in range(future_days):
        
        pred = model.predict(input_seq, verbose=0)
        
        # Store prediction
        future_predictions.append(pred[0, 0])
        
        # Reshape prediction properly to (1,1,1)
        pred_reshaped = pred.reshape(1, 1, 1)
        
        # Remove first timestep and append prediction
        input_seq = np.concatenate(
            (input_seq[:, 1:, :], pred_reshaped),
            axis=1
        )

    # Convert back to original scale
    future_predictions = scaler.inverse_transform(
        np.array(future_predictions).reshape(-1, 1)
    )

    return future_predictions.flatten()

In [15]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

In [16]:
time_steps = 30

# --- SARIMA ---
sarima_model = SARIMAX(train['Close'],
                       order=(4,0,3),
                       seasonal_order=(1,0,1,5))

sarima_result = sarima_model.fit()

forecast = sarima_result.forecast(steps=len(test))

# --- Residuals ---
residuals = test['Close'].values - forecast.values
residuals = residuals.reshape(-1,1)

# --- Scaling ---
from sklearn.preprocessing import StandardScaler
scaler_res = StandardScaler()
res_scaled = scaler_res.fit_transform(residuals)

X_res, y_res = create_sequences(res_scaled, time_steps)

# --- RNN ---
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

model = Sequential([
    SimpleRNN(61, return_sequences=True, input_shape=(time_steps,1)),
    Dropout(0.2),
    SimpleRNN(32),
    Dense(1)
])

model.compile(optimizer=Adam(0.0005), loss='huber')

early_stop = EarlyStopping(patience=10, restore_best_weights=True)

model.fit(X_res, y_res,
          epochs=150,
          batch_size=16,
          callbacks=[early_stop],
          verbose=0)

# --- Prediction ---
pred = model.predict(X_res)
pred = scaler_res.inverse_transform(pred)

final_pred = forecast[time_steps:].values + pred.flatten()
actual = test['Close'].values[time_steps:]

7/7 [==============================] - 0s 3ms/step


In [17]:
future_days = 20
future_prices = predict_future_prices(
    model,
    test_scaled,
    scaler,
    time_steps,
    future_days
)

print("Future Prices:", future_prices)

Future Prices: [25370.193 25235.543 25119.49  25060.598 25046.207 24983.51  24880.17
 24723.68  24635.932 24718.908 24782.047 24744.037 24725.77  24820.568
 25018.027 25071.777 25097.395 25078.018 25172.84  25288.717]


In [18]:
last_date = df.index[-1]
future_dates = pd.date_range(last_date, periods=future_days+1, freq='29D')[1:]

future_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Price": future_prices
})

print(future_df)

         Date  Predicted_Price
0  2026-03-21     25370.193359
1  2026-04-19     25235.542969
2  2026-05-18     25119.490234
3  2026-06-16     25060.597656
4  2026-07-15     25046.207031
5  2026-08-13     24983.509766
6  2026-09-11     24880.169922
7  2026-10-10     24723.679688
8  2026-11-08     24635.931641
9  2026-12-07     24718.908203
10 2027-01-05     24782.046875
11 2027-02-03     24744.037109
12 2027-03-04     24725.769531
13 2027-04-02     24820.568359
14 2027-05-01     25018.027344
15 2027-05-30     25071.777344
16 2027-06-28     25097.394531
17 2027-07-27     25078.017578
18 2027-08-25     25172.839844
19 2027-09-23     25288.716797


In [19]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 simple_rnn_2 (SimpleRNN)    (None, 30, 61)            3843      
                                                                 
 dropout_1 (Dropout)         (None, 30, 61)            0         
                                                                 
 simple_rnn_3 (SimpleRNN)    (None, 32)                3008      
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 6884 (26.89 KB)
Trainable params: 6884 (26.89 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [20]:
# Create DataFrame
future_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Price": future_prices
})

# --------------------------------------------------
# 🔥 NEW COLUMN: Price Change (Difference)
# --------------------------------------------------

# Difference between consecutive days
future_df['Change'] = future_df['Predicted_Price'].diff()

# First value fix (no previous day)
future_df['Change'].iloc[0] = 0

# --------------------------------------------------
# 🔥 NEW COLUMN: Up/Down Indicator
# --------------------------------------------------

future_df['Movement'] = future_df['Change'].apply(
    lambda x: f"\033[32m+{round(x,2)} ▲\033[0m" if x > 0 else (f"\033[31m{round(x,2)} ▼\033[0m" if x < 0 else "0")
)

print(tabulate(future_df, headers='keys', tablefmt='pretty', showindex=False))

+---------------------+-----------------+----------------+-----------+
|        Date         | Predicted_Price |     Change     | Movement  |
+---------------------+-----------------+----------------+-----------+
| 2026-03-21 00:00:00 | 25370.193359375 |      0.0       |     0     |
| 2026-04-19 00:00:00 | 25235.54296875  | -134.650390625 | -134.65 ▼ |
| 2026-05-18 00:00:00 | 25119.490234375 | -116.052734375 | -116.05 ▼ |
| 2026-06-16 00:00:00 | 25060.59765625  | -58.892578125  | -58.89 ▼  |
| 2026-07-15 00:00:00 | 25046.20703125  |   -14.390625   | -14.39 ▼  |
| 2026-08-13 00:00:00 | 24983.509765625 | -62.697265625  |  -62.7 ▼  |
| 2026-09-11 00:00:00 | 24880.169921875 | -103.33984375  | -103.34 ▼ |
| 2026-10-10 00:00:00 |  24723.6796875  | -156.490234375 | -156.49 ▼ |
| 2026-11-08 00:00:00 | 24635.931640625 | -87.748046875  | -87.75 ▼  |
| 2026-12-07 00:00:00 | 24718.908203125 |   82.9765625   | +82.98 ▲  |
| 2027-01-05 00:00:00 |  24782.046875   |  63.138671875  | +63.14 ▲  |
| 2027

## 1) Trend :
#### uptrend/downtrend 

## 2) Volatality
#### Market risk / fluctuation

## 3) Movement:
#### Change, % change, arrows

## 4) Returns:
#### Profit / loss %